
# Forward-Looking Regime Detection Using LSI and IRI for Dynamic Portfolio Allocation

**FA800 QWIM Project Notebook**  
**Project Theme:** Bank of America: Market Regimes, Changepoints, Bubbles and Crashes in QWIM  

This notebook presents the Python coding framework for our FA800 QWIM project. The main objective is to build a forward-looking regime detection and allocation system using two stress indicators: the Liquidity Stress Index (LSI) and the Implied Risk Index (IRI). These indicators help capture different dimensions of market stress before they fully appear in portfolio returns.

The project begins with weekly financial data loading and cleaning, followed by exploratory data analysis and statistical testing. After preparing the data, we construct standardized stress features and use them to identify market regimes such as calm periods, elevated stress, liquidity stress, and crisis-like environments. The regime signals are then linked to portfolio allocation decisions across major ETFs such as SPY, TLT, GLD, and HYG.

The purpose of this notebook is not only to run models, but also to show how each part of the framework supports the final investment decision. Therefore, the notebook follows the full project workflow: data retrieval, data analysis, model testing, regime detection, benchmark comparison, allocation analysis, result interpretation, and report/dashboard-ready output generation.

In [ ]:
#!pip install pandas numpy matplotlib scipy scikit-learn hmmlearn statsmodels

  Using cached hmmlearn-0.3.3-cp312-cp312-macosx_10_9_universal2.whl.metadata (3.0 kB)
Using cached hmmlearn-0.3.3-cp312-cp312-macosx_10_9_universal2.whl (196 kB)


## 1. Imports and Setup


This section imports all Python libraries required for the project. The notebook mainly uses `pandas` and `numpy` for data handling, `matplotlib` for visualizations, `scikit-learn` for preprocessing, PCA, and clustering, and `scipy` for portfolio optimization.

The setup also defines basic project settings such as warning handling, random seed, chart resolution, and output folder creation. Keeping these settings in one place makes the notebook easier to run, debug, and reproduce.

In [3]:
from __future__ import annotations

import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import jarque_bera, skew, kurtosis
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["figure.dpi"] = 120

try:
    from hmmlearn.hmm import GaussianHMM
    HMM_AVAILABLE = True
except Exception:
    GaussianHMM = None
    HMM_AVAILABLE = False

try:
    from statsmodels.tsa.stattools import adfuller
    STATSMODELS_AVAILABLE = True
except Exception:
    adfuller = None
    STATSMODELS_AVAILABLE = False

print("HMM available:", HMM_AVAILABLE)
print("ADF test available:", STATSMODELS_AVAILABLE)

HMM available: True
ADF test available: True



## 2. Project Configuration

This section defines the main project settings before the analysis begins. I set the data directory, output folder, core ETF universe, and regime names in one place so that the rest of the notebook can use the same settings consistently.

The core ETF universe includes SPY, TLT, GLD, and HYG. These assets were selected because they represent different market exposures: equities, Treasury bonds, gold, and high-yield credit. This makes them useful for testing whether regime signals can improve portfolio allocation across both risk-on and defensive assets.

The regime labels are ordered from the lowest-stress environment to the highest-stress environment. This makes the results easier to interpret later when we compare market behavior, portfolio weights, and strategy performance across regimes.

In [ ]:
# Define the main project paths and modeling choices.
# Keeping these settings together makes the notebook easier to update if the data folder, asset universe, or regime labels change later.
DATA_DIR = Path(".").resolve()
OUTPUT_DIR = DATA_DIR / "outputs_notebook"
OUTPUT_DIR.mkdir(exist_ok=True)

# Core ETF universe used for regime-aware allocation:
# SPY = U.S. equities, TLT = long-term Treasuries,
# GLD = gold, HYG = high-yield corporate credit.

CORE_TICKERS = ["SPY", "TLT", "GLD", "HYG"]

# Regime names are arranged from low stress to high stress.
# The raw model labels are statistical, so later we map them into economically meaningful labels using average LSI and IRI stress levels.

REGIME_NAMES = {
    0: "Deep Calm",
    1: "Calm",
    2: "Elevated Stress",
    3: "Crisis",
}

print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

TypeError: unsupported operand type(s) for /: 'str' and 'str'

## 3. Data Loading Functions

This section creates reusable functions for loading the weekly ETF price files used in the project. Each ETF file is read, cleaned, converted into a proper time-series format, and aligned to a weekly Friday date index.

The reason for writing this as a function is to avoid repeating the same cleaning steps for SPY, TLT, GLD, and HYG separately. Once the function works for one ETF, it can be applied consistently to all assets in the portfolio universe.

The second function combines all individual ETF price series into one aligned price panel. This panel becomes the base dataset for calculating weekly returns, running benchmark strategies, and testing regime-aware portfolio allocation.

In [1]:

def load_weekly_etf_csv(path: Path, ticker: str) -> pd.Series:
    # Load a yfinance-style weekly CSV and return clean weekly Close prices.
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path.name}")

# This function loads one weekly ETF CSV file and returns a clean price series. It handles the yfinance-style file format, converts the date column properly, keeps the Close price, and aligns each observation to a Friday weekly date.

    raw = pd.read_csv(path)

# Some yfinance exports contain extra header rows before the actual data. If that format appears, we skip those metadata rows and manually assign column names.

    if "Price" in raw.columns:
        df = pd.read_csv(
            path,
            skiprows=3,
            header=None,
            names=["Date", "Close", "High", "Low", "Open", "Volume"],
        )
    else:
        df = raw.copy()
        if "Date" not in df.columns:
            df = df.rename(columns={df.columns[0]: "Date"})
        if "Close" not in df.columns:
            raise ValueError(f"{path.name} must contain a Close column.")

# Convert dates and prices into usable numeric time-series fields. Invalid rows are removed so the final series is clean.

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
    df = df.dropna(subset=["Date", "Close"])

    # Align observations to Friday weekly dates.
    df["Date"] = df["Date"] + pd.to_timedelta((4 - df["Date"].dt.weekday) % 7, unit="D")
    df = df.set_index("Date").sort_index()
    df.index = df.index.tz_localize(None)
    return df["Close"].rename(ticker)


def load_prices(data_dir: Path, tickers: List[str]) -> pd.DataFrame:
    # Combine all ETF prices into one aligned price panel.
    prices = pd.concat(
        [load_weekly_etf_csv(data_dir / f"{ticker}_weekly.csv", ticker) for ticker in tickers],
        axis=1,
    ).dropna()
    if prices.empty:
        raise RuntimeError("ETF price panel is empty after loading files.")
    return prices

NameError: name 'Path' is not defined

## 4. Load ETF Prices and Compute Returns

This section loads the weekly price data for the core ETF universe and converts those prices into weekly returns. The portfolio model cannot work directly with price levels because each ETF has a different price scale. Returns make the assets comparable and allow us to measure portfolio performance consistently.

The price panel contains the weekly closing prices for SPY, TLT, GLD, and HYG. After loading the prices, percentage changes are calculated to create the weekly return panel. These returns are later used for benchmark construction, regime-conditional performance analysis, optimization, and strategy backtesting.

In [ ]:
# Load the aligned weekly closing price panel for the selected ETF universe.
prices = load_prices(DATA_DIR, CORE_TICKERS)

# Convert weekly prices into weekly percentage returns.
# Returns are used instead of prices because portfolio performance depends on percentage gains/losses.
returns = prices.pct_change().dropna()

# Print the dimensions of the price and return panels to confirm the data loaded correctly.
print("Price panel shape:", prices.shape)
print("Return panel shape:", returns.shape)

# Display the first few rows so we can visually inspect the dataset before modeling.
display(prices.head())
display(returns.head())

## 5. Data Quality Checks and Descriptive Statistics

This section checks whether the ETF data is clean and suitable for modeling. Before running any regime model or portfolio strategy, it is important to confirm the date coverage, number of observations, and missing values for each asset.

The descriptive statistics summarize the weekly return behavior of SPY, TLT, GLD, and HYG. In addition to the usual mean and standard deviation, I also calculate skewness and excess kurtosis. These two measures are important in financial data because asset returns are often not normally distributed. Negative skewness may indicate downside risk, while high excess kurtosis suggests fat tails and large extreme movements.

This step connects directly to the project motivation: financial markets do not behave in a stable linear way across all periods. The presence of skewness, fat tails, and changing volatility supports the need for regime-based analysis instead of relying only on static portfolio assumptions.

In [ ]:
# Check missing values, first available date, last available date, and number of valid observations for each ETF price series.
quality_table = pd.DataFrame({
    "missing_values": prices.isna().sum(),
    "first_date": prices.apply(lambda s: s.dropna().index.min()),
    "last_date": prices.apply(lambda s: s.dropna().index.max()),
    "observations": prices.notna().sum(),
})

# Create a return summary table. The usual describe() output gives mean, standard deviation, min, max, and quartiles.
return_summary = returns.describe().T

# Add skewness to understand whether returns are tilted toward large gains or large losses.
return_summary["skewness"] = returns.apply(skew)

# Add excess kurtosis to check for fat tails. Positive excess kurtosis means extreme return observations are more common than in a normal distribution.
return_summary["excess_kurtosis"] = returns.apply(lambda x: kurtosis(x, fisher=True))

# Display both tables for inspection.
display(quality_table)
display(return_summary)

## 6. Exploratory Data Analysis

This section gives an initial visual understanding of how the ETF universe behaved over the full sample period. The growth-of-$1 chart shows how one dollar invested in each ETF would have grown through time. This helps compare long-term performance across equities, bonds, gold, and high-yield credit.

The correlation matrix shows how weekly returns move together across the four assets. This is important for portfolio construction because diversification depends not only on individual asset returns, but also on how assets behave relative to each other. If correlations rise during stress periods, a static portfolio may become less diversified exactly when protection is needed most.

These EDA plots support the project argument that asset behavior changes across market conditions, which motivates the next steps: stress index construction, regime detection, and regime-aware allocation.

In [ ]:
# Convert weekly returns into cumulative growth of $1. This helps us compare how each ETF performed over the full sample.
growth = (1 + returns).cumprod()

# Plot cumulative growth for the core ETF universe.
ax = growth.plot(title="ETF Growth of $1")
ax.set_ylabel("Growth of $1")
plt.tight_layout()

# Save the chart so it can be used later in the report or dashboard.
plt.savefig(OUTPUT_DIR / "eda_growth_of_1.png")
plt.show()

# Calculate the weekly return correlation matrix.
# Correlation helps us understand diversification across the selected assets.
corr = returns.corr()
display(corr)

# Plot the correlation matrix as a heatmap-style visual. This makes cross-asset relationships easier to interpret visually.
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values)

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns)
ax.set_yticklabels(corr.index)

ax.set_title("Weekly Return Correlation Matrix")

# Add correlation values inside each cell.
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")

fig.colorbar(im, ax=ax)
plt.tight_layout()

# Save the correlation chart for report/dashboard use.
plt.savefig(OUTPUT_DIR / "eda_correlation_matrix.png")
plt.show()

## 7. Statistical Tests

This section adds formal statistical tests before moving into regime modeling. The goal is to check whether the return series behave like stable normal distributions or whether they show statistical patterns that justify more advanced regime-based analysis.

The Augmented Dickey-Fuller test is used to check stationarity. In this project, stationarity is important because most time-series and portfolio models assume that the statistical behavior of returns is reasonably stable over the sample. A low ADF p-value usually suggests that the series is stationary.

The Jarque-Bera test is used to check whether the return distribution is normally distributed. Financial return data often violates normality because it contains skewness and fat tails. If the Jarque-Bera p-value is very small, it indicates that the return series is not normally distributed. This supports our project motivation because market returns often behave differently during calm and crisis periods.

In [ ]:
# Function to run the Augmented Dickey-Fuller test. This test checks whether a time series is stationary. A lower p-value usually supports stationarity.
def run_adf_test(series: pd.Series) -> Dict[str, float]:
    clean = series.dropna()
    if not STATSMODELS_AVAILABLE:
        return {"ADF Statistic": np.nan, "p-value": np.nan}
    stat, pval, *_ = adfuller(clean)
    return {"ADF Statistic": stat, "p-value": pval}


# Function to run the Jarque-Bera test. This test checks whether a return series follows a normal distribution. A small p-value suggests non-normality, which is common in financial returns.
def run_jb_test(series: pd.Series) -> Dict[str, float]:
    clean = series.dropna()
    stat, pval = jarque_bera(clean)
    return {"JB Statistic": stat, "p-value": pval}


# Run ADF and Jarque-Bera tests for each ETF return series. I also include skewness and excess kurtosis so the normality result can be interpreted more clearly.
stat_rows = []

for col in returns.columns:
    adf = run_adf_test(returns[col])
    jb = run_jb_test(returns[col])

    stat_rows.append({
        "asset": col,
        "ADF Statistic": adf["ADF Statistic"],
        "ADF p-value": adf["p-value"],
        "JB Statistic": jb["JB Statistic"],
        "JB p-value": jb["p-value"],
        "skewness": skew(returns[col].dropna()),
        "excess_kurtosis": kurtosis(returns[col].dropna(), fisher=True),
    })


# Convert test results into a table and save it for report/dashboard use.
test_results = pd.DataFrame(stat_rows).set_index("asset")

display(test_results)

test_results.to_csv(OUTPUT_DIR / "statistical_tests.csv")

## 8. Load or Construct LSI and IRI

This section creates the two main regime-driving variables for the project: the Liquidity Stress Index (LSI) and the Implied Risk Index (IRI). These two indices are the core inputs for detecting market regimes.

If `indices_weekly.csv` is available, the notebook directly loads LSI and IRI from that file. This is preferred because it uses the stress-index construction already prepared for the project. If the file is not available, the notebook creates a fallback version using ETF return behavior, rolling volatility, and safe-haven relationships. The fallback keeps the notebook runnable, but the local `indices_weekly.csv` version should be used for the final project results.

The purpose of this step is to move from raw market data into cleaner stress signals. LSI represents market liquidity and funding stress, while IRI represents implied or perceived risk in the market. These two signals are later used to classify regimes such as calm, elevated stress, and crisis-like environments.

In [ ]:
# This function loads the project's stress indices if indices_weekly.csv is available.
# The preferred input is a local file containing LSI and IRI because those indices
# are directly connected to the project methodology and presentation.
def load_or_build_stress_indices(data_dir: Path, returns: pd.DataFrame) -> pd.DataFrame:
    idx_path = data_dir / "indices_weekly.csv"

    # First, try to use the prepared project stress-index file.
    # This keeps the notebook aligned with the LSI/IRI framework used in the slides.
    if idx_path.exists():
        idx = pd.read_csv(idx_path, index_col=0, parse_dates=True)
        idx.index = pd.to_datetime(idx.index, errors="coerce").tz_localize(None)

        # Check whether both required columns are present.
        # If yes, keep only LSI and IRI and align them to Friday weekly dates.
        if {"LSI", "IRI"}.issubset(idx.columns):
            stress = idx[["LSI", "IRI"]].apply(pd.to_numeric, errors="coerce").dropna()
            stress = stress.resample("W-FRI").last().dropna()
            print("Using local indices_weekly.csv for LSI and IRI.")
            return stress

        print("indices_weekly.csv exists but LSI and IRI were not both found. Building fallback stress indices.")

    # Fallback section.
    # This is only used when the prepared LSI/IRI file is missing or incomplete.
    # It creates simple stress proxies from ETF return behavior so the notebook still runs.
    print("Building fallback LSI/IRI from ETF return behavior.")

    features = pd.DataFrame(index=returns.index)

    # Equity stress proxy: large absolute SPY movements indicate unstable market behavior.
    features["SPY_abs_return"] = returns["SPY"].abs()

    # Volatility proxy: rolling SPY volatility captures changing market uncertainty.
    features["SPY_rolling_vol"] = returns["SPY"].rolling(4).std()

    # Credit stress proxy: negative HYG returns indicate pressure in high-yield credit.
    features["HYG_stress"] = -returns["HYG"].rolling(4).mean()

    # Safe-haven proxy: TLT often performs defensively when risk assets weaken.
    features["TLT_safe_haven"] = returns["TLT"].rolling(4).mean()

    # Gold proxy: GLD can act as a defensive or crisis-sensitive asset.
    features["GLD_safe_haven"] = returns["GLD"].rolling(4).mean()

    features = features.dropna()

    # Standardize features before PCA so no single feature dominates due to scale.
    x = StandardScaler().fit_transform(features)

    # Use two principal components as fallback stress factors.
    # These are named LSI and IRI to keep the rest of the notebook workflow consistent.
    pcs = PCA(n_components=2, random_state=42).fit_transform(x)

    stress = pd.DataFrame(pcs, index=features.index, columns=["LSI", "IRI"])

    # Align the sign of LSI so higher values correspond more naturally to higher volatility stress.
    if np.corrcoef(stress["LSI"], features["SPY_rolling_vol"])[0, 1] < 0:
        stress["LSI"] *= -1

    return stress


# Load the prepared LSI/IRI file or construct fallback stress indices if needed.
stress = load_or_build_stress_indices(DATA_DIR, returns)

# Align stress indices with the available ETF return dates.
stress = stress.loc[stress.index.intersection(returns.index)].dropna()

print("Stress index shape:", stress.shape)
display(stress.head())

##9. Rolling Z-Scores and PCA View

This section transforms LSI and IRI into rolling z-scores using a 52-week window. A rolling z-score compares the current stress level with its own recent one-year history. This makes the stress indicators easier to interpret because it tells us whether current stress is normal, elevated, or unusually high compared with the recent market environment.

A z-score close to 0 means stress is near its recent average. A positive z-score means stress is above normal. Values above 2 are usually treated as unusually high stress or crisis-like conditions. This logic is useful for identifying bubbles, crashes, and stress transitions.

The PCA view is included to show how much of the variation in LSI and IRI can be summarized by principal components. PCA helps confirm whether the two stress indicators share a common systemic stress factor while still preserving separate liquidity and implied-risk information.

Rolling z-scores show when stress is normal, elevated, or crisis-like. PCA helps explain how latent stress factors summarize correlated stress inputs.

In [ ]:
# Function to calculate rolling z-scores. A rolling z-score compares each value with its own recent 52-week mean and standard deviation.
def rolling_zscore(df: pd.DataFrame, window: int = 52) -> pd.DataFrame:
    mean = df.rolling(window).mean()
    std = df.rolling(window).std()
    return ((df - mean) / std).dropna()


# Convert LSI and IRI into 52-week rolling z-scores. This makes the two stress indicators easier to compare and interpret.
stress_z = rolling_zscore(stress, window=52)
stress_z.columns = ["LSI_Z", "IRI_Z"]


# Plot rolling z-scores for LSI and IRI. The horizontal line at 2 marks an unusually elevated stress threshold. The horizontal line at 0 marks normal recent average stress.
ax = stress_z.plot(title="Rolling Z-Scores: LSI_Z and IRI_Z")
ax.axhline(2, linestyle="--", linewidth=1)
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_ylabel("Rolling Z-Score")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "stress_rolling_zscores.png")
plt.show()


# Standardize raw LSI and IRI before PCA. PCA is scale-sensitive, so standardization is required.
stress_scaled = StandardScaler().fit_transform(stress[["LSI", "IRI"]])


# Run PCA on the two stress indices. This gives a simple view of how much common variation is captured by each component.
pca = PCA(n_components=2, random_state=42)
stress_pcs = pca.fit_transform(stress_scaled)


# Summarize how much variance each principal component explains.
pca_summary = pd.DataFrame({
    "component": ["PC1", "PC2"],
    "explained_variance_ratio": pca.explained_variance_ratio_,
})


# Show PCA loadings. Loadings help explain whether the component is driven more by LSI, IRI, or both.
loadings = pd.DataFrame(
    pca.components_.T,
    index=["LSI", "IRI"],
    columns=["PC1", "PC2"],
)


display(pca_summary)
display(loadings)

## 10. Regime Detection Functions

This section defines the functions used to detect market regimes from LSI and IRI. The preferred model is the Hidden Markov Model because the project assumes that market regimes are not directly observed. Instead, regimes are hidden states that generate the observed stress indicators.

The HMM is useful because it captures three important features of financial regimes: persistence, transition probabilities, and uncertainty. Markets usually do not jump randomly from calm to crisis and back every week. Stress conditions tend to persist and then transition gradually. This makes HMM more suitable than a simple clustering model for the main project framework.

A Gaussian Mixture Model is included as a fallback option. GMM can classify observations into groups based on stress behavior, but it does not model regime persistence through a transition matrix. Therefore, GMM is useful for comparison or backup, while HMM remains the main regime engine.

After the raw model labels are created, the regimes are reordered from low stress to high stress using the average values of LSI and IRI. This step is necessary because statistical models assign arbitrary labels such as 0, 1, 2, and 3. Reordering makes the results economically interpretable as Deep Calm, Calm, Elevated Stress, and Crisis.

In [ ]:
# Fit the regime detection model using LSI and IRI. The preferred model is HMM because regimes are hidden, persistent, and probabilistic. If hmmlearn is not installed, the notebook usesGaussian Mixture Model as a fallback so the workflow still runs.
def fit_regime_model(stress_df: pd.DataFrame, n_states: int = 4, seed: int = 42):
    x = stress_df[["LSI", "IRI"]].dropna().values

    if HMM_AVAILABLE:
        # Hidden Markov Model:
        # This estimates hidden regimes and the probability of moving
        # from one regime to another over time.
        model = GaussianHMM(
            n_components=n_states,
            covariance_type="full",
            n_iter=500,
            random_state=seed
        )
        model.fit(x)
        raw_labels = model.predict(x)
        probs = model.predict_proba(x)
        model_type = "GaussianHMM"

    else:
        # Fallback model:
        # GMM clusters the stress observations but does not directly model
        # regime transition dynamics like an HMM.
        model = GaussianMixture(
            n_components=n_states,
            covariance_type="full",
            random_state=seed
        )
        model.fit(x)
        raw_labels = model.predict(x)
        probs = model.predict_proba(x)
        model_type = "GaussianMixture fallback"

    idx = stress_df.dropna().index

    # Store raw statistical labels before economic relabeling.
    raw_regimes = pd.Series(raw_labels, index=idx, name="raw_regime")

    # Store regime probabilities for each raw state.
    prob_df = pd.DataFrame(probs, index=idx)

    return model, raw_regimes, prob_df, model_type


# Reorder raw regime labels from low-stress to high-stress. This makes the model output easier to explain in economic terms.
def order_regimes(raw_regimes: pd.Series, stress_df: pd.DataFrame) -> Tuple[pd.Series, Dict[int, int]]:

    # Combine raw labels with stress indicators.
    tmp = pd.concat([raw_regimes, stress_df[["LSI", "IRI"]]], axis=1).dropna()

    # Calculate average stress level for each raw regime. A simple combined stress score is used here: average LSI + average IRI.
    stress_score = tmp.groupby("raw_regime").apply(
        lambda g: (g["LSI"] + g["IRI"]).mean()
    )

    # Sort raw regimes from lowest average stress to highest average stress.
    ordered_raw_states = stress_score.sort_values().index.tolist()

    # Create mapping: raw label → ordered economic label.
    # Example: raw state 2 may become ordered regime 0 if it has the lowest stress.
    mapping = {
        int(raw): int(rank)
        for rank, raw in enumerate(ordered_raw_states)
    }

    # Apply mapping to get economically ordered regime labels.
    ordered = raw_regimes.map(mapping).astype(int).rename("regime")

    return ordered, mapping


# Reorder probability columns so they match the economically ordered regimes. This ensures regime_0 means low stress and regime_3 means highest stress.
def order_probability_columns(
    prob_df: pd.DataFrame,
    mapping: Dict[int, int],
    n_states: int = 4
) -> pd.DataFrame:

    ordered = pd.DataFrame(
        0.0,
        index=prob_df.index,
        columns=[f"regime_{i}" for i in range(n_states)]
    )

    # Move each raw-state probability into its ordered regime column.
    for raw_state, ordered_state in mapping.items():
        ordered[f"regime_{ordered_state}"] = prob_df.iloc[:, raw_state]

    # Normalize rows so probabilities sum to 1.
    row_sums = ordered.sum(axis=1).replace(0, np.nan)
    return ordered.div(row_sums, axis=0).fillna(1.0 / n_states)

## 11. Fit and Interpret Regimes

This section fits the regime detection model using the LSI and IRI stress indicators. The model first produces raw statistical regime labels. Since these raw labels do not automatically have economic meaning, they are reordered from low stress to high stress based on the average values of LSI and IRI.

After reordering, the four regimes are interpreted as Deep Calm, Calm, Elevated Stress, and Crisis. This makes the output easier to connect with the market story. Deep Calm represents low liquidity stress and low implied risk, while Crisis represents periods where both stress indicators are highly elevated.

The regime table created here becomes one of the most important outputs of the notebook. It is later used for regime-conditional return analysis, portfolio weight selection, benchmark comparison, and dashboard/report generation.

In [ ]:
# Number of market regimes used in the project.
# We use four states to capture a range from calm conditions to crisis-like stress.
N_STATES = 4

# Fit the preferred regime model using LSI and IRI.
# If HMM is available, this uses GaussianHMM; otherwise it uses GMM as a fallback.
model, raw_regimes, raw_probs, model_type = fit_regime_model(
    stress,
    n_states=N_STATES
)

# Reorder the raw statistical labels into economically meaningful labels.
# The final ordering is from low stress to high stress.
regimes, regime_mapping = order_regimes(raw_regimes, stress)

# Reorder probability columns so regime_0, regime_1, regime_2, and regime_3
# match the same low-to-high stress interpretation.
regime_probs = order_probability_columns(
    raw_probs,
    regime_mapping,
    n_states=N_STATES
)

# Combine stress indices and final regime labels into one table.
# This table is useful for interpretation, plotting, dashboarding, and report outputs.
regime_table = pd.concat([stress, regimes], axis=1).dropna()

# Add readable regime names for presentation and interpretation.
regime_table["regime_name"] = regime_table["regime"].map(REGIME_NAMES)

# Print model information so we know whether HMM or fallback GMM was used.
print("Model used:", model_type)
print("Raw-to-ordered regime mapping:", regime_mapping)

# Count how many weeks fall into each regime.
# This helps us understand whether one regime dominates the sample.
display(regime_table["regime_name"].value_counts().to_frame("weeks"))

# Summarize average LSI and IRI behavior by regime.
# This confirms whether the economic ordering makes sense.
display(
    regime_table
    .groupby("regime_name")[["LSI", "IRI"]]
    .agg(["mean", "std", "count"])
)

# Save regime classifications and probabilities for later report/dashboard use.
regime_table.to_csv(OUTPUT_DIR / "regime_classification.csv")
regime_probs.to_csv(OUTPUT_DIR / "regime_probabilities.csv")

## 12. Regime Visualizations

This section visualizes the detected regimes in three ways. First, the time-series plot shows how the Liquidity Stress Index changes over time and colors each observation by the detected regime. This helps connect the statistical regime labels with historical stress episodes.

Second, the regime-space plot shows LSI on one axis and IRI on the other axis. This view helps explain how the model separates market environments based on the joint behavior of liquidity stress and implied risk. Calm regimes should generally appear in the lower-stress region, while crisis-like regimes should appear where both indicators are elevated.

Third, the probability plot shows how the model’s confidence in each regime changes through time. This is important because the model does not only produce a hard classification. It also gives regime probabilities, which are more useful for forward-looking allocation decisions.

In [ ]:
# Plot LSI through time and color the points by detected regime. This helps us see when the model identifies calm, elevated stress, or crisis-like periods.
fig, ax = plt.subplots(figsize=(12, 5))

for regime_id, name in REGIME_NAMES.items():
    subset = regime_table[regime_table["regime"] == regime_id]
    ax.scatter(
        subset.index,
        subset["LSI"],
        s=9,
        label=name
    )

ax.set_title("LSI Over Time Colored by Detected Regime")
ax.set_ylabel("LSI")
ax.legend(ncol=2, fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "regime_lsi_timeline.png")
plt.show()


# Plot the regime space using LSI and IRI. This shows how regimes are separated based on the joint stress environment.
fig, ax = plt.subplots(figsize=(8, 6))

for regime_id, name in REGIME_NAMES.items():
    subset = regime_table[regime_table["regime"] == regime_id]
    ax.scatter(
        subset["LSI"],
        subset["IRI"],
        s=12,
        label=name,
        alpha=0.75
    )

ax.set_title("Regime Space: LSI vs IRI")
ax.set_xlabel("Liquidity Stress Index (LSI)")
ax.set_ylabel("Implied Risk Index (IRI)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "regime_space_lsi_iri.png")
plt.show()


# Rename probability columns from regime_0, regime_1, etc. into readable regime names. This makes the probability chart easier to interpret in the presentation and report.
prob_plot = regime_probs.rename(
    columns={f"regime_{i}": REGIME_NAMES[i] for i in range(N_STATES)}
)

# Plot regime probabilities through time. These probabilities show uncertainty and regime transition risk.
ax = prob_plot.plot(
    title="Regime Probabilities Through Time",
    figsize=(12, 5)
)

ax.set_ylabel("Probability")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "regime_probabilities.png")
plt.show()

## 13. Portfolio Optimization Functions

This section defines the portfolio optimization functions used in the allocation stage. The goal is to convert regime information into actual portfolio weights across SPY, TLT, GLD, and HYG.

The max-Sharpe optimizer is used for the main regime-optimized strategy. It selects weights that maximize expected return per unit of risk, using the estimated return and covariance matrix for each regime. To keep the solution realistic, the optimizer is long-only, fully invested, and capped at 60% in any single asset.

The minimum-volatility and risk-parity functions are included as benchmark methods. Minimum volatility focuses on reducing portfolio risk, while risk parity spreads risk more evenly across assets. These benchmarks help us compare whether the regime-aware strategy adds value beyond standard portfolio construction methods.

In [ ]:
# Max-Sharpe optimizer.
# This function finds the portfolio weights that maximize return per unit of risk. It is used for the main regime-optimized allocation strategy.
def max_sharpe_weights(mu: np.ndarray, cov: np.ndarray, max_single: float = 0.60) -> np.ndarray:
    n = len(mu)

    # Objective function:
    # scipy minimizes functions, so we minimize the negative Sharpe ratio.
    def objective(w):
        ret = float(w @ mu)
        vol = float(np.sqrt(w @ cov @ w))
        return -ret / (vol + 1e-12)

    # Constraint: portfolio weights must sum to 1. This keeps the portfolio fully invested.
    constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)

    # Bounds: long-only weights with a 60% maximum allocation to any one asset.
    # This prevents unrealistic concentration in one ETF.
    bounds = [(0.0, max_single)] * n

    # Start with equal weights as the initial guess.
    w0 = np.ones(n) / n

    # Solve the constrained optimization problem.
    result = minimize(
        objective,
        w0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 500}
    )

    # If optimization fails, return equal weights as a safe fallback.
    return result.x if result.success else w0


# Minimum-volatility optimizer.
# This benchmark focuses only on lowering portfolio variance.
def min_vol_weights(cov: np.ndarray, max_single: float = 0.60) -> np.ndarray:

    # Fast approximate minimum-variance weighting using inverse covariance. Negative weights are clipped to preserve the long-only project constraint.
    n = cov.shape[0]
    inv_cov = np.linalg.pinv(cov)
    ones = np.ones(n)

    weights = inv_cov @ ones
    weights = np.clip(weights, 0, max_single)

    # If all weights are removed after clipping, revert to equal weight.
    if weights.sum() <= 0:
        weights = np.ones(n) / n
    else:
        weights = weights / weights.sum()

    return weights


# Risk-parity benchmark.
# This approximate version gives more weight to lower-volatility assets. It is faster and simpler for a classroom notebook than solving a nonlinear program every week.
def risk_parity_weights(cov: np.ndarray, max_single: float = 0.60) -> np.ndarray:

    # Estimate asset volatilities from the covariance matrix.
    vols = np.sqrt(np.diag(cov))

    # Inverse-volatility weighting is a common practical approximation to risk parity.
    inv_vol = 1 / np.maximum(vols, 1e-12)

    weights = inv_vol / inv_vol.sum()

    # Apply the same 60% concentration cap used in the main strategy.
    weights = np.minimum(weights, max_single)

    return weights / weights.sum()

## 14. Phase II Hand Weights and Optimized Regime Weights

This section compares two approaches to regime-based portfolio allocation. The first approach uses Phase II hand-picked weights based on economic intuition. These weights reflect how we expect different assets to behave across calm, moderate, elevated-stress, and crisis regimes.

The second approach estimates optimized regime weights from the historical return behavior within each detected regime. For each regime, the notebook calculates regime-conditional expected returns and covariance, then applies a max-Sharpe optimization with a 60% concentration cap. Ledoit-Wolf covariance shrinkage is used to reduce the instability that can occur when covariance estimates are noisy or when a regime has fewer observations.

This comparison is important because it separates economic intuition from data-driven optimization. The hand weights are easier to explain and may be more stable, while optimized weights are more systematic and directly estimated from regime-specific asset behavior.

In [ ]:
# Phase II hand-picked regime weights.
# These weights reflect economic intuition about how assets should behave in each regime. Rows are regimes and columns are ETFs.
hand_weights = pd.DataFrame(
    {
        "SPY": [0.65, 0.55, 0.25, 0.10],
        "TLT": [0.10, 0.20, 0.40, 0.35],
        "GLD": [0.10, 0.10, 0.25, 0.45],
        "HYG": [0.15, 0.15, 0.10, 0.10],
    },
    index=[REGIME_NAMES[i] for i in range(N_STATES)],
)


# Estimate optimized weights separately for each detected regime. The optimizer uses regime-conditional returns and Ledoit-Wolf covariance shrinkage.
def estimate_regime_optimized_weights(
    returns_df: pd.DataFrame,
    regimes_s: pd.Series,
    max_single: float = 0.60,
    min_obs: int = 20
) -> pd.DataFrame:

    # Align returns and regimes on the same dates.
    aligned_returns, aligned_regimes = returns_df.align(regimes_s, join="inner", axis=0)

    rows = []

    # Estimate one optimized weight vector for each regime.
    for regime_id in range(N_STATES):

        # Select returns observed during the current regime.
        sample = aligned_returns[aligned_regimes == regime_id].dropna()

        # If a regime has too few observations, use equal weights. This avoids overfitting to a very small sample.
        if len(sample) < min_obs:
            w = np.ones(aligned_returns.shape[1]) / aligned_returns.shape[1]

        else:
            # Annualize mean weekly returns.
            mu = sample.mean().values * 52

            # Estimate annualized covariance using Ledoit-Wolf shrinkage. Shrinkage makes the covariance matrix more stable.
            cov = LedoitWolf().fit(sample.values).covariance_ * 52

            # Solve max-Sharpe weights with long-only and concentration constraints.
            w = max_sharpe_weights(mu, cov, max_single=max_single)

        rows.append(w)

    # Return weights as a readable table with regime names and ETF columns.
    return pd.DataFrame(
        rows,
        index=[REGIME_NAMES[i] for i in range(N_STATES)],
        columns=returns_df.columns
    )


# Compute the optimized regime weights.
optimized_weights = estimate_regime_optimized_weights(returns, regimes)

# Display Phase II hand weights in percentage format.
print("Phase II hand weights")
display((hand_weights * 100).round(1).astype(str) + "%")

# Display optimized weights in percentage format.
print("Post-Phase-II optimized weights")
display((optimized_weights * 100).round(1).astype(str) + "%")

# Save both weight tables for the report and dashboard.
hand_weights.to_csv(OUTPUT_DIR / "phase2_hand_weights.csv")
optimized_weights.to_csv(OUTPUT_DIR / "phase3_optimized_weights.csv")

## 15. Backtesting Functions

This section defines the backtesting functions used to evaluate the portfolio strategies. The purpose of the backtest is to simulate how each strategy would have performed through time using weekly ETF returns.

The first function runs a regime-based strategy. It selects portfolio weights based on the detected regime for each week. This allows us to test whether regime-specific weights improve performance compared with static benchmarks.

The second function runs a constant-mix benchmark, such as a 60/40 portfolio. This is important because the regime-aware strategy should be compared against a simple and widely understood baseline.

The third function runs expanding-window optimized benchmarks such as risk parity and minimum volatility. These benchmarks are recomputed using only historical data available up to that point, which makes the comparison more realistic. Transaction costs and turnover are also included so that the results do not overstate performance.

In [ ]:
# This function backtests a regime-based strategy.
# For each week, the portfolio selects weights based on the detected market regime.
# It includes transaction costs and a drift-based rebalancing rule.

def run_strategy_from_regime_weights(
    returns_df,
    regimes_s,
    weight_table,
    tc_bps=10.0,
    rebalance_threshold=0.03
):
    # Align returns and regime labels on the same dates.
    aligned_returns, aligned_regimes = returns_df.align(regimes_s, join="inner", axis=0)

    # Start with zero portfolio weights before the first rebalance.
    current_w = np.zeros(aligned_returns.shape[1])

    # Store weekly portfolio returns, turnover, and weight history.
    port_rets = []
    turnovers = []
    weights_history = []

    for date in aligned_returns.index:

        # Identify the regime for the current week.
        regime_name = REGIME_NAMES[int(aligned_regimes.loc[date])]

        # Select the target weights linked to that regime.
        target = weight_table.loc[regime_name].values.astype(float)

        # Check how far the current weights have drifted from the target weights.
        # If this drift is larger than the threshold, the strategy rebalances.
        drift = np.abs(current_w - target).max() if current_w.sum() > 0 else 1.0

        if drift > rebalance_threshold:
            # Turnover measures the total absolute change in portfolio weights.
            trade = np.abs(target - current_w).sum()

            # Transaction cost is charged based on turnover.
            cost = trade * (tc_bps / 10000.0)

            # Update portfolio weights to target weights after rebalancing.
            current_w = target.copy()

        else:
            # If no rebalance is needed, no trading cost is applied.
            trade = 0.0
            cost = 0.0

        # Calculate weekly portfolio return after transaction cost.
        week_return = float(current_w @ aligned_returns.loc[date].values - cost)

        port_rets.append(week_return)
        turnovers.append(trade)
        weights_history.append(current_w.copy())

        # Let weights drift after the weekly returns are realized.
        grown = current_w * (1 + aligned_returns.loc[date].values)

        # Normalize weights so they sum to 1 again.
        current_w = grown / grown.sum() if grown.sum() != 0 else target.copy()

    return (
        pd.Series(port_rets, index=aligned_returns.index, name="strategy_return"),
        pd.Series(turnovers, index=aligned_returns.index, name="turnover"),
        pd.DataFrame(weights_history, index=aligned_returns.index, columns=aligned_returns.columns)
    )

In [ ]:
# This function backtests a static constant-mix benchmark.
# The same target weights are used throughout the full sample.
# Examples include 60/40 or equal-weight portfolios.

def run_constant_mix(
    returns_df,
    weights,
    tc_bps=10.0,
    rebalance_threshold=0.03
):
    # Start with zero weights before the first rebalance.
    current_w = np.zeros(len(weights))

    # Store weekly portfolio returns, turnover, and weight history.
    port_rets = []
    turnovers = []
    weights_history = []

    for date in returns_df.index:

        # Static target weights stay the same every week.
        target = weights.astype(float)

        # Measure how far current weights have drifted from target weights.
        drift = np.abs(current_w - target).max() if current_w.sum() > 0 else 1.0

        if drift > rebalance_threshold:
            # Turnover is the total absolute weight change.
            trade = np.abs(target - current_w).sum()

            # Transaction cost is deducted whenever a rebalance occurs.
            cost = trade * (tc_bps / 10000.0)

            # Reset portfolio to target weights.
            current_w = target.copy()

        else:
            # No rebalance means no turnover and no trading cost.
            trade = 0.0
            cost = 0.0

        # Calculate weekly benchmark return after transaction cost.
        week_return = float(current_w @ returns_df.loc[date].values - cost)

        port_rets.append(week_return)
        turnovers.append(trade)
        weights_history.append(current_w.copy())

        # Allow portfolio weights to drift naturally after returns occur.
        grown = current_w * (1 + returns_df.loc[date].values)

        # Normalize weights after market movement.
        current_w = grown / grown.sum() if grown.sum() != 0 else target.copy()

    return (
        pd.Series(port_rets, index=returns_df.index, name="benchmark_return"),
        pd.Series(turnovers, index=returns_df.index, name="turnover"),
        pd.DataFrame(weights_history, index=returns_df.index, columns=returns_df.columns)
    )

In [ ]:
# This function backtests expanding-window optimized benchmarks.
# The benchmark weights are updated using only past data.
# This avoids look-ahead bias because future returns are not used to form current weights.

def run_expanding_optimizer(
    returns_df,
    mode="risk_parity",
    min_train=156,
    tc_bps=10.0,
    rebalance_threshold=0.03,
    optimize_every=4
):
    # Start with zero weights.
    current_w = np.zeros(returns_df.shape[1])

    # Initial target is equal weight until enough training data is available.
    target = np.ones(returns_df.shape[1]) / returns_df.shape[1]

    # Store weekly returns, turnover, weight history, and output dates.
    port_rets = []
    turnovers = []
    weights_history = []
    dates = []

    for i in range(min_train, len(returns_df)):

        # Recompute benchmark weights every few weeks. Between optimization dates, the previous target is carried forward.
        if (i == min_train) or ((i - min_train) % optimize_every == 0):

            # Use only historical data available before the current date.
            hist = returns_df.iloc[:i].dropna()

            # Estimate annualized covariance using Ledoit-Wolf shrinkage.
            cov = LedoitWolf().fit(hist.values).covariance_ * 52

            # Choose which optimized benchmark to run.
            if mode == "risk_parity":
                target = risk_parity_weights(cov)

            elif mode == "min_vol":
                target = min_vol_weights(cov)

            else:
                raise ValueError("mode must be either 'risk_parity' or 'min_vol'")

        # Current date being tested.
        date = returns_df.index[i]

        # Check whether the current portfolio has drifted too far from the target.
        drift = np.abs(current_w - target).max() if current_w.sum() > 0 else 1.0

        if drift > rebalance_threshold:
            # Compute turnover and transaction cost.
            trade = np.abs(target - current_w).sum()
            cost = trade * (tc_bps / 10000.0)
            current_w = target.copy()

        else:
            # No rebalance needed.
            trade = 0.0
            cost = 0.0

        # Calculate weekly return after transaction cost.
        week_return = float(current_w @ returns_df.loc[date].values - cost)

        port_rets.append(week_return)
        turnovers.append(trade)
        weights_history.append(current_w.copy())
        dates.append(date)

        # Allow weights to drift after weekly returns are realized.
        grown = current_w * (1 + returns_df.loc[date].values)

        # Normalize weights to keep the portfolio fully invested.
        current_w = grown / grown.sum() if grown.sum() != 0 else target.copy()

    return (
        pd.Series(port_rets, index=dates, name=f"{mode}_return"),
        pd.Series(turnovers, index=dates, name="turnover"),
        pd.DataFrame(weights_history, index=dates, columns=returns_df.columns)
    )

## 16. Run All Strategies

This section runs all portfolio strategies under the same backtesting assumptions. The purpose is to compare static benchmarks, regime-aware hand weights, optimized regime weights, and risk-based optimized benchmarks on a fair basis.

The first 156 weeks are used as a training/burn-in window. This avoids starting the backtest before the model has enough historical information. A 10 bps transaction cost is applied whenever the portfolio trades, and a 3% drift threshold is used to decide when rebalancing is necessary.

The strategies compared here are: 60/40 benchmark, equal weight, Phase II hand-weight regime strategy, post-Phase-II optimized regime strategy, risk parity, and minimum volatility. This gives a full comparison between simple static portfolios, economically motivated regime allocation, and systematic optimized allocation methods.

In [ ]:
# Main backtest settings.
# MIN_TRAIN gives the model a 156-week burn-in window before evaluating performance.
# TC_BPS applies transaction costs in basis points whenever the strategy rebalances.
# REBALANCE_THRESHOLD means the strategy only trades when portfolio weights drift by more than 3%.
MIN_TRAIN = 156
TC_BPS = 10.0
REBALANCE_THRESHOLD = 0.03


# Align returns and regimes on the same date index.
# This ensures every strategy is evaluated over the same available weekly observations.
common_index = returns.index.intersection(regimes.index)

returns_common = returns.loc[common_index]
regimes_common = regimes.loc[common_index]


# Create the out-of-sample testing window.
# The first 156 weeks are skipped so that the model has enough historical data.
returns_oos = returns_common.iloc[MIN_TRAIN:]
regimes_oos = regimes_common.iloc[MIN_TRAIN:]


# Run the Phase II hand-weight regime strategy.
# This uses the intuition-based weights assigned to each regime.
hand_ret, hand_to, hand_w_hist = run_strategy_from_regime_weights(
    returns_oos,
    regimes_oos,
    hand_weights,
    TC_BPS,
    REBALANCE_THRESHOLD
)


# Run the post-Phase-II optimized regime strategy.
# This uses regime-conditional optimized weights estimated from historical data.
opt_ret, opt_to, opt_w_hist = run_strategy_from_regime_weights(
    returns_oos,
    regimes_oos,
    optimized_weights,
    TC_BPS,
    REBALANCE_THRESHOLD
)


# Run the 60/40 benchmark.
# This benchmark allocates 60% to SPY and 40% to TLT.
bench_6040_ret, bench_6040_to, bench_6040_w = run_constant_mix(
    returns_oos,
    np.array([0.60, 0.40, 0.00, 0.00]),
    TC_BPS,
    REBALANCE_THRESHOLD
)


# Run the equal-weight benchmark.
# This gives 25% allocation to each ETF.
equal_ret, equal_to, equal_w = run_constant_mix(
    returns_oos,
    np.ones(len(CORE_TICKERS)) / len(CORE_TICKERS),
    TC_BPS,
    REBALANCE_THRESHOLD
)


# Run the expanding-window risk parity benchmark.
# This benchmark updates weights using only past data.
rp_ret, rp_to, rp_w = run_expanding_optimizer(
    returns_common,
    mode="risk_parity",
    min_train=MIN_TRAIN,
    tc_bps=TC_BPS,
    rebalance_threshold=REBALANCE_THRESHOLD
)


# Run the expanding-window minimum-volatility benchmark.
# This benchmark also uses only past data to estimate the covariance matrix.
mv_ret, mv_to, mv_w = run_expanding_optimizer(
    returns_common,
    mode="min_vol",
    min_train=MIN_TRAIN,
    tc_bps=TC_BPS,
    rebalance_threshold=REBALANCE_THRESHOLD
)


# Combine all strategy return series into one table.
# This makes performance comparison easier in later sections.
strategy_returns = pd.DataFrame({
    "60/40 Benchmark": bench_6040_ret,
    "Equal Weight": equal_ret,
    "Phase II Hand + Regime": hand_ret,
    "Post-Phase-II Regime Optimized": opt_ret,
    "Risk Parity": rp_ret,
    "Minimum Volatility": mv_ret,
}).dropna()


# Combine strategy turnover series into one table.
# Turnover is important because high trading activity can reduce real-world performance.
strategy_turnover = pd.DataFrame({
    "60/40 Benchmark": bench_6040_to,
    "Equal Weight": equal_to,
    "Phase II Hand + Regime": hand_to,
    "Post-Phase-II Regime Optimized": opt_to,
    "Risk Parity": rp_to,
    "Minimum Volatility": mv_to,
}).reindex(strategy_returns.index).fillna(0)


# Confirm that the combined return table was created correctly.
print("Strategy return panel shape:", strategy_returns.shape)

# Display first few rows for a quick check.
display(strategy_returns.head())

## 17. Performance Summary

This section creates the main performance scorecard for all strategies. The purpose is to compare each portfolio on both return and risk metrics instead of looking only at total return.

The key metrics include total return, annualized return, annualized volatility, Sharpe ratio, maximum drawdown, Calmar ratio, and annual turnover. Total return and annualized return show growth. Volatility and maximum drawdown show risk. Sharpe and Calmar ratios summarize risk-adjusted performance. Turnover is included because a strategy with high trading activity may look good before costs but become less practical after transaction costs.

This scorecard is one of the most important outputs of the project because it directly supports the final investment conclusion and benchmark comparison.

In [ ]:
# Function to calculate the main portfolio performance statistics.
# These metrics are used to compare all strategies on return, risk, drawdown, and turnover.
def performance_stats(
    ret: pd.Series,
    turnover: pd.Series | None = None,
    freq: int = 52
) -> Dict[str, float]:

    # Remove missing values from the return series.
    ret = ret.dropna()

    # Cumulative growth of $1 invested in the strategy.
    growth = (1 + ret).cumprod()

    # Total return over the full backtest window.
    total_return = growth.iloc[-1] - 1

    # Annualized return converts total growth into a yearly rate.
    ann_return = growth.iloc[-1] ** (freq / len(ret)) - 1

    # Annualized volatility scales weekly return volatility to a yearly number.
    ann_vol = ret.std() * np.sqrt(freq)

    # Sharpe ratio measures return earned per unit of volatility.
    # A higher Sharpe ratio means stronger risk-adjusted performance.
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan

    # Maximum drawdown measures the largest peak-to-trough loss.
    # This is important because investors care about downside risk, not only average return.
    max_dd = (growth / growth.cummax() - 1).min()

    # Calmar ratio compares annualized return with maximum drawdown.
    # It helps evaluate whether the return was worth the worst observed loss.
    calmar = ann_return / abs(max_dd) if max_dd < 0 else np.nan

    # Annual turnover shows how much the portfolio trades per year.
    # This matters because high turnover can make a strategy less practical after costs.
    ann_turnover = (
        np.nan
        if turnover is None
        else turnover.loc[ret.index].sum() / (len(ret) / freq)
    )

    return {
        "Total Return": total_return,
        "Ann Return": ann_return,
        "Ann Vol": ann_vol,
        "Sharpe": sharpe,
        "Max Drawdown": max_dd,
        "Calmar": calmar,
        "Ann Turnover": ann_turnover,
    }


# Calculate performance statistics for every strategy.
summary = pd.DataFrame({
    name: performance_stats(
        strategy_returns[name],
        strategy_turnover[name]
    )
    for name in strategy_returns.columns
}).T


# Create a display version with percentages formatted clearly.
summary_display = summary.copy()

for col in ["Total Return", "Ann Return", "Ann Vol", "Max Drawdown", "Ann Turnover"]:
    summary_display[col] = (summary_display[col] * 100).round(2).astype(str) + "%"

for col in ["Sharpe", "Calmar"]:
    summary_display[col] = summary_display[col].round(3)


# Display the final headline performance scorecard.
display(summary_display)


# Save raw numerical results and strategy-level time series for report/dashboard use.
summary.to_csv(OUTPUT_DIR / "headline_summary.csv")
strategy_returns.to_csv(OUTPUT_DIR / "strategy_weekly_returns.csv")
strategy_turnover.to_csv(OUTPUT_DIR / "strategy_turnover.csv")

## 18. Growth and Drawdown Charts

This section creates two presentation-ready charts for strategy comparison. The first chart shows the growth of $1 invested in each strategy. This helps compare long-term portfolio performance across the regime-aware strategies and benchmarks.

The second chart shows strategy drawdowns through time. Drawdown is important because it shows how much the portfolio falls from its previous peak. A strategy with strong returns but very deep drawdowns may be difficult to hold in real investment conditions.

Together, these charts help evaluate both the upside and downside behavior of each strategy. They are also useful for the final report and dashboard because they visually explain the trade-off between growth, risk, and crisis protection.

In [ ]:
# Convert weekly strategy returns into cumulative growth of $1.
# This shows how one dollar invested in each strategy would have grown through time.
strategy_growth = (1 + strategy_returns).cumprod()

# Plot strategy growth paths.
# This chart compares the long-term performance of regime strategies and benchmarks.
ax = strategy_growth.plot(
    title="Strategy Growth of $1: Regime Strategies vs Benchmarks",
    figsize=(12, 5)
)

ax.set_ylabel("Growth of $1")

plt.tight_layout()

# Save the chart for the written report, presentation, or dashboard.
plt.savefig(OUTPUT_DIR / "strategy_growth.png")

plt.show()


# Calculate drawdowns for each strategy.
# Drawdown = current portfolio value divided by previous peak minus 1.
# This shows the worst decline from peak value at each point in time.
strategy_drawdowns = strategy_growth / strategy_growth.cummax() - 1

# Plot strategy drawdowns.
# This chart helps compare downside risk across strategies.
ax = strategy_drawdowns.plot(
    title="Strategy Drawdowns",
    figsize=(12, 5)
)

ax.set_ylabel("Drawdown")

plt.tight_layout()

# Save the drawdown chart for report/dashboard use.
plt.savefig(OUTPUT_DIR / "strategy_drawdowns.png")

plt.show()

## 19. Conditional Asset Behavior by Regime

This section analyzes how each asset behaves inside each detected market regime. Instead of looking at average returns over the entire sample, the notebook separates returns by regime and calculates annualized return and annualized volatility for SPY, TLT, GLD, and HYG.

This is important because the main idea of the project is that asset behavior changes across market environments. For example, SPY and HYG may perform better in calm regimes, while TLT and GLD may become more useful during elevated stress or crisis regimes. By studying returns and volatility within each regime, we can check whether the regime labels have real economic meaning.

These conditional statistics also support the portfolio allocation step. If asset return and risk characteristics differ across regimes, then a static allocation may not be optimal across all market states.

In [ ]:
# Align weekly ETF returns with the final ordered regime labels.
# This ensures that each return observation is matched with its detected market regime.
aligned_returns, aligned_regimes = returns.align(regimes, join="inner", axis=0)


# Store conditional return and volatility statistics for each asset inside each regime.
conditional_rows = []


# Loop through each regime and calculate asset behavior within that state.
for regime_id, regime_name in REGIME_NAMES.items():

    # Select only the return observations that occurred during the current regime.
    sample = aligned_returns.loc[aligned_regimes == regime_id]

    # Calculate conditional statistics for each ETF.
    for asset in aligned_returns.columns:

        conditional_rows.append({
            "regime": regime_name,
            "asset": asset,

            # Number of weeks observed in this regime.
            "weeks": len(sample),

            # Annualized return within the regime.
            "ann_return": sample[asset].mean() * 52,

            # Annualized volatility within the regime.
            "ann_vol": sample[asset].std() * np.sqrt(52),
        })


# Convert the collected rows into a clean table.
conditional_asset_stats = pd.DataFrame(conditional_rows)


# Display annualized returns by regime and asset.
# This helps identify which assets tend to perform best in each regime.
display(
    conditional_asset_stats.pivot(
        index="regime",
        columns="asset",
        values="ann_return"
    )
)


# Display annualized volatility by regime and asset.
# This helps compare how asset risk changes across market regimes.
display(
    conditional_asset_stats.pivot(
        index="regime",
        columns="asset",
        values="ann_vol"
    )
)


# Save the conditional asset statistics for the final report and dashboard.
conditional_asset_stats.to_csv(
    OUTPUT_DIR / "conditional_asset_stats_by_regime.csv",
    index=False
)

## 20. Transaction Cost Sensitivity

This section tests whether the main strategies remain useful after applying different transaction cost assumptions. This is important because regime-based strategies may rebalance more often than static benchmarks. If the turnover is too high, even a strong strategy can lose its advantage after trading costs.

The analysis compares strategy performance at 0 bps, 10 bps, 25 bps, and 50 bps transaction cost levels. A lower cost assumption represents a more liquid or institutional trading environment, while higher cost assumptions represent more conservative real-world trading conditions.

This step helps evaluate whether the regime framework is practically deployable. A strategy is stronger if it continues to perform well even when transaction costs increase.

In [ ]:
# Transaction cost levels to test.
# These are measured in basis points.
# 0 bps = no trading cost, 10 bps = base case, 25 and 50 bps = higher-cost stress tests.
cost_levels = [0, 10, 25, 50]

# Store results from each transaction cost scenario.
cost_rows = []


# Run the three main strategies at each transaction cost level.
# This helps test whether performance survives after realistic trading frictions.
for cost in cost_levels:

    # Phase II hand-weight regime strategy under the selected cost level.
    h_ret, h_to, _ = run_strategy_from_regime_weights(
        returns_oos,
        regimes_oos,
        hand_weights,
        cost,
        REBALANCE_THRESHOLD
    )

    # Post-Phase-II optimized regime strategy under the selected cost level.
    o_ret, o_to, _ = run_strategy_from_regime_weights(
        returns_oos,
        regimes_oos,
        optimized_weights,
        cost,
        REBALANCE_THRESHOLD
    )

    # 60/40 benchmark under the selected cost level.
    b_ret, b_to, _ = run_constant_mix(
        returns_oos,
        np.array([0.60, 0.40, 0.00, 0.00]),
        cost,
        REBALANCE_THRESHOLD
    )

    # Calculate performance statistics for each strategy and cost level.
    for name, ret, to in [
        ("60/40 Benchmark", b_ret, b_to),
        ("Phase II Hand + Regime", h_ret, h_to),
        ("Post-Phase-II Regime Optimized", o_ret, o_to),
    ]:
        row = performance_stats(ret, to)
        row["strategy"] = name
        row["Transaction Cost bps"] = cost
        cost_rows.append(row)


# Combine all cost-sensitivity results into one table.
cost_sensitivity = pd.DataFrame(cost_rows)


# Display selected metrics that matter most for cost analysis.
display(
    cost_sensitivity[
        [
            "Transaction Cost bps",
            "strategy",
            "Total Return",
            "Ann Return",
            "Ann Vol",
            "Sharpe",
            "Max Drawdown",
            "Ann Turnover",
        ]
    ]
)


# Save transaction cost sensitivity results for final report and dashboard.
cost_sensitivity.to_csv(
    OUTPUT_DIR / "cost_sensitivity.csv",
    index=False
)

## 21. Report and Dashboard Output Files

This final section saves the main project outputs in a format that can be used for the written report, presentation, and interactive dashboard. The notebook has already generated the strategy returns, turnover, growth paths, drawdowns, regime labels, and regime probabilities. This section organizes those outputs into clean CSV files.

The section also creates a short result interpretation that identifies which strategy has the best Sharpe ratio, lowest volatility, strongest drawdown profile, and highest total return. This is useful for the final report because it converts the numerical scorecard into a clear investment conclusion.

The dashboard-ready files can later be connected to a Shiny or Streamlit dashboard. For example, the dashboard can use the regime data to show regime probabilities over time and use the strategy data to show growth curves, drawdowns, turnover, and performance comparisons.

In [ ]:
# Identify the strongest strategy under each main evaluation metric.
# These outputs help summarize the project results in a report-friendly way.

best_sharpe = summary["Sharpe"].idxmax()
best_drawdown = summary["Max Drawdown"].idxmax()
best_return = summary["Total Return"].idxmax()
lowest_vol = summary["Ann Vol"].idxmin()


# Create a short written interpretation of the results.
# This can be copied directly into the presentation, written report, or dashboard notes.
interpretation = f"""
### Result Interpretation

The regime framework translates market stress indicators into portfolio decisions.
The best Sharpe ratio is achieved by **{best_sharpe}**, while the strongest maximum drawdown profile is achieved by **{best_drawdown}**.
The highest total return is achieved by **{best_return}**, showing that raw return and risk-adjusted performance are not always the same.
The lowest annualized volatility is achieved by **{lowest_vol}**.

Overall, the notebook supports the project argument that regime detection adds value when it improves risk-adjusted performance, drawdown control, or portfolio interpretability relative to static benchmarks.
"""


# Print the interpretation so it is visible inside the notebook.
print(interpretation)


# Save the interpretation as a markdown file.
# This file can be reused in the written report or presentation appendix.
with open(OUTPUT_DIR / "presentation_interpretation.md", "w") as f:
    f.write(interpretation)


# Build a dashboard-ready regime dataset.
# This combines stress indicators, regime probabilities, and final regime labels.
dashboard_regime_data = pd.concat(
    [
        stress,
        regime_probs,
        regimes.rename("regime")
    ],
    axis=1
).dropna()


# Add readable regime names for dashboard filters and charts.
dashboard_regime_data["regime_name"] = dashboard_regime_data["regime"].map(REGIME_NAMES)


# Build a dashboard-ready strategy dataset.
# This combines returns, turnover, cumulative growth, and drawdowns in one file.
dashboard_strategy_data = pd.concat(
    {
        "returns": strategy_returns,
        "turnover": strategy_turnover,
        "growth": strategy_growth,
        "drawdown": strategy_drawdowns,
    },
    axis=1
)


# Save dashboard-ready files.
dashboard_regime_data.to_csv(OUTPUT_DIR / "dashboard_regime_data.csv")
dashboard_strategy_data.to_csv(OUTPUT_DIR / "dashboard_strategy_data.csv")


# Print the saved output folder and list of generated CSV files.
print("Saved outputs to:", OUTPUT_DIR)

for file in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", file.name)